# 01. Check Input And Reference Annotations

This notebook is the setup and sanity-check step before running `insituCNV`.

Use it to:

- load a small example dataset from the publication, or replace it with your own `.h5ad`
- confirm that spatial coordinates are present
- confirm that raw counts are available
- inspect the annotation column that will define the normal reference cells for `infercnvpy`

Important assumption:

Your dataset should already contain a column in `adata.obs` that marks normal or healthy reference cells. This can be a cell-type label or a region annotation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Example dataset from the publication.
# Replace this path with your own .h5ad file when needed.
DATA_PATH = Path('/home/augusta/storage3/augusta/insituCNV/InSituCNV/Breast_cancer_Xenium5K/01_InSituCNV/data/1105_BL.h5ad')

# Choose the annotation column that contains normal/reference labels.
# This can be a healthy cell-type annotation or a region annotation.
REFERENCE_KEY = 'cell_type_oct25'

# Edit this list to match the healthy or normal categories in your own dataset.
REFERENCE_CATEGORIES = ['T_cells', 'B_cells', 'Myeloid', 'Plasma', 'Fibroblast', 'Endothelial', 'Adipocytes', 'PVLs']

OUTPUT_DIR = PROJECT_ROOT / 'results' / 'notebook_example_1105_BL'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = str(OUTPUT_DIR / 'figures')
Path(sc.settings.figdir).mkdir(parents=True, exist_ok=True)

## Load The Dataset

If your dataset already has `raw_counts`, `spatial`, and a normal annotation column, you can use it directly here.

In [ ]:
adata = sc.read_h5ad(DATA_PATH)
adata

## Check Required Fields

This block makes the minimum requirements explicit.

- `adata.obsm['spatial']` is needed for spatial plots
- `adata.layers['raw_counts']` is preferred for smoothing and CNV inference
- `REFERENCE_KEY` must exist in `adata.obs`

In [ ]:
if 'spatial' not in adata.obsm:
    raise KeyError("adata.obsm['spatial'] is required.")

if REFERENCE_KEY not in adata.obs.columns:
    raise KeyError(f"{REFERENCE_KEY!r} was not found in adata.obs.")

if 'raw_counts' not in adata.layers:
    print("raw_counts layer was not found. Copying current X into adata.layers['raw_counts'].")
    adata.layers['raw_counts'] = adata.X.copy()

print('n_cells =', adata.n_obs)
print('n_genes =', adata.n_vars)
print('reference_key =', REFERENCE_KEY)

## Inspect The Annotation Column

The values below tell you whether your chosen reference labels are actually present in the dataset.

In [ ]:
adata.obs[REFERENCE_KEY].value_counts(dropna=False)

In [ ]:
present_reference_categories = [cat for cat in REFERENCE_CATEGORIES if cat in set(adata.obs[REFERENCE_KEY].astype(str))]
present_reference_categories

## Visualize The Chosen Reference Annotation In Space

This is often the most important manual check before running CNV inference.

Make sure the categories you plan to use as reference cells look biologically reasonable.

In [ ]:
sc.pl.embedding(
    adata,
    basis='spatial',
    color=REFERENCE_KEY,
    title=f'Spatial plot: {REFERENCE_KEY}',
    frameon=False,
    size=12,
)

## Optional: Create A Binary Region-Style Reference Label

If your dataset uses a region annotation instead of cell types, you can create a simplified label here.

For example, you could map several normal compartments into a single `normal_reference` label.

In [ ]:
# Example only. Edit or skip this cell if you do not need it.
# adata.obs['reference_region'] = adata.obs[REFERENCE_KEY].astype(str).map(
#     lambda x: 'normal_reference' if x in REFERENCE_CATEGORIES else 'other'
# )
# adata.obs['reference_region'].value_counts()

## Save The Checked Input Object

This makes it easy to start the CNV notebook from a clean checkpoint.

In [ ]:
checked_path = OUTPUT_DIR / 'adata_checked_input.h5ad'
adata.write(checked_path, compression='gzip')
print(checked_path)